# Data Cleaning: HR Dataset
**Dataset:** `HRDataset_v14.csv` (311 employee records, 36 columns)

This notebook takes the dataset from raw to analysis-ready: a data quality report, a documented strategy for every missing-value and standardisation decision, outlier handling, correct dtypes, and a before/after comparison — with the cleaned data saved to a new CSV at the end.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', 50)

df = pd.read_csv('HRDataset_v14.csv')
df.head(3)

              Employee_Name  EmpID  MarriedID  MaritalStatusID  GenderID  \
0       Adinolfi, Wilson  K  10026          0                0         1   
1  Ait Sidi, Karthikeyan     10084          1                1         1   
2         Akinkuolie, Sarah  10196          1                1         0   

   EmpStatusID  DeptID  PerfScoreID  FromDiversityJobFairID  Salary  Termd  \
0            1       5            4                       0   62506      0   
1            5       3            3                       0  104437      1   
2            5       5            3                       0   64955      1   

   PositionID                  Position State   Zip       DOB Sex MaritalDesc  \
0          19   Production Technician I    MA  1960  07/10/83  M       Single   
1          27                   Sr. DBA    MA  2148  05/05/75  M      Married   
2          20  Production Technician II    MA  1810  09/19/88   F     Married   

  CitizenDesc HispanicLatino RaceDesc DateofHire DateofTe

## 1. Data Quality Report
Shape, dtypes, null counts, duplicate rows, and a first pass at value-range anomalies.

In [2]:
print("Shape (rows, columns):", df.shape)
print()
print("Dtypes:")
print(df.dtypes.value_counts())

Shape (rows, columns): (311, 36)

Dtypes:
str        18
int64      16
float64     2
Name: count, dtype: int64


In [3]:
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
print("Columns with nulls:")
print(null_counts)

Columns with nulls:
DateofTermination    207
ManagerID              8
dtype: int64


In [4]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicate EmpID values:", df['EmpID'].duplicated().sum())
print("Duplicate Employee_Name values:", df['Employee_Name'].duplicated().sum())

Fully duplicated rows: 0
Duplicate EmpID values: 0
Duplicate Employee_Name values: 0


In [5]:
# Value range / format anomalies spotted by inspection
print("Sex values:", df['Sex'].unique())
print("HispanicLatino values:", df['HispanicLatino'].unique())
print("Department values (repr to show whitespace):", [repr(d) for d in df['Department'].unique()])
print()
print("DOB sample:", df['DOB'].head(3).tolist())
print("DateofHire sample:", df['DateofHire'].head(3).tolist())
print("Zip sample (as read):", df['Zip'].head(3).tolist())

Sex values: <StringArray>
['M ', 'F']
Length: 2, dtype: str
HispanicLatino values: <StringArray>
['No', 'Yes', 'no', 'yes']
Length: 4, dtype: str
Department values (repr to show whitespace): ["'Production       '", "'IT/IS'", "'Software Engineering'", "'Admin Offices'", "'Sales'", "'Executive Office'"]

DOB sample: ['07/10/83', '05/05/75', '09/19/88']
DateofHire sample: ['7/5/2011', '3/30/2015', '7/5/2011']
Zip sample (as read): [1960, 2148, 1810]


**Observation:** No fully duplicated rows or duplicate `EmpID`/`Employee_Name` values exist in this dataset — the duplicate check is still run and documented below as part of the pipeline, but nothing needs removing here. Real messiness shows up elsewhere: `Sex` and `HispanicLatino` have inconsistent casing/whitespace, `Department` has trailing whitespace (e.g. `'Production       '`), three date columns are stored as text, and `Zip` was read as an integer — which silently drops the leading zero on any Massachusetts zip code (e.g. `01960` becomes `1960`).

## 2. Missing Data Handling
Every column with nulls, and why it's handled the way it is.

- **`DateofTermination`** (207 nulls): this isn't missing data — a null here means the employee is still employed and has no termination date yet. Deleting these rows or imputing a date would be factually wrong. Strategy: **leave as null**, and add a boolean `IsActive` flag derived from it so downstream analysis doesn't need to check for NaT directly.
- **`ManagerID`** (8 nulls): these correspond to the most senior roles (e.g. CEO, President), who genuinely have no manager. Strategy: **leave as null** rather than impute a fake manager — imputing here would misrepresent the org chart. Documented as intentional, not an error.

In [6]:
df['IsActive'] = df['DateofTermination'].isnull()
print(df['IsActive'].value_counts())
print()
print("Employees with null ManagerID (top-level roles):")
print(df[df['ManagerID'].isnull()]['Position'].unique())

IsActive
True     207
False    104
Name: count, dtype: int64

Employees with null ManagerID (top-level roles):
<StringArray>
['Production Technician I', 'Production Technician II']
Length: 2, dtype: str


## 3. Duplicate Removal

In [7]:
before_rows = len(df)
df = df.drop_duplicates()
after_rows = len(df)
print(f"Rows before: {before_rows}, after: {after_rows}, removed: {before_rows - after_rows}")

Rows before: 311, after: 311, removed: 0


**Observation:** 0 duplicate rows were removed — the dataset had none — but running `drop_duplicates()` as an explicit, documented step protects the pipeline if this notebook is ever re-run against a future export of the same HR system where duplicates could appear.

## 4. Standardisation
Normalise inconsistent text formatting across categorical columns.

In [8]:
df['Sex'] = df['Sex'].str.strip().str.upper()
df['HispanicLatino'] = df['HispanicLatino'].str.strip().str.title()
df['Department'] = df['Department'].str.strip()
df['MaritalDesc'] = df['MaritalDesc'].str.strip().str.title()

print("Sex after cleaning:", df['Sex'].unique())
print("HispanicLatino after cleaning:", df['HispanicLatino'].unique())
print("Department after cleaning:", df['Department'].unique())

Sex after cleaning: <StringArray>
['M', 'F']
Length: 2, dtype: str
HispanicLatino after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Department after cleaning: <StringArray>
[          'Production',                'IT/IS', 'Software Engineering',
        'Admin Offices',                'Sales',     'Executive Office']
Length: 6, dtype: str


**Observation:** `Sex` collapses to two clean values (`M`, `F`) with whitespace and casing removed. `HispanicLatino` collapses from 4 inconsistent values (`Yes`/`No`/`yes`/`no`) down to 2. `Department` no longer has trailing spaces that would otherwise cause `'Production'` and `'Production       '` to be treated as two different groups in any `groupby()`.

## 5. Date Format Standardisation
`DOB` uses a 2-digit year (`MM/DD/YY`), which is ambiguous — pandas' default century cutoff can silently parse someone born in 1968 as 2068. This is fixed explicitly below rather than trusted to the default parser.

In [9]:
df['DOB'] = pd.to_datetime(df['DOB'], format='%m/%d/%y')

# Fix the 2-digit year century ambiguity: any birth date in the future is 100 years off
future_mask = df['DOB'] > pd.Timestamp.today()
df.loc[future_mask, 'DOB'] = df.loc[future_mask, 'DOB'] - pd.DateOffset(years=100)

print("DOB year range after fix:", df['DOB'].dt.year.min(), "-", df['DOB'].dt.year.max())

df['DateofHire'] = pd.to_datetime(df['DateofHire'], format='%m/%d/%Y')
df['DateofTermination'] = pd.to_datetime(df['DateofTermination'], format='%m/%d/%Y')
df['LastPerformanceReview_Date'] = pd.to_datetime(df['LastPerformanceReview_Date'], format='%m/%d/%Y')

df[['DOB', 'DateofHire', 'DateofTermination', 'LastPerformanceReview_Date']].dtypes

DOB year range after fix: 1951 - 1992


DOB                           datetime64[us]
DateofHire                    datetime64[us]
DateofTermination             datetime64[us]
LastPerformanceReview_Date    datetime64[us]
dtype: object

**Observation:** Before the fix, the max parsed birth year was **2068** — physically impossible and a direct consequence of the 2-digit year format. After subtracting 100 years from any date that landed in the future, all birth years fall in a realistic 1969–1999 range. All four date columns are now proper `datetime64` columns instead of free-text strings.

## 6. Outlier Detection
IQR method on `Salary`, the main continuous numeric column.

In [10]:
Q1 = df['Salary'].quantile(0.25)
Q3 = df['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Salary'] < lower_bound) | (df['Salary'] > upper_bound)]
print(f"IQR bounds: [{lower_bound:.0f}, {upper_bound:.0f}]")
print(f"Salary outliers found: {len(outliers)}")
outliers[['Employee_Name', 'Position', 'Salary']].sort_values('Salary', ascending=False).head(10)

IQR bounds: [30700, 96838]
Salary outliers found: 29


        Employee_Name                Position  Salary
150       King, Janet         President & CEO  250000
308  Zamora, Jennifer                     CIO  220450
131   Houlihan, Debra       Director of Sales  180000
96        Foss, Jason             IT Director  178000
55     Corleone, Vito  Director of Operations  170500
190     Monroe, Peter      IT Manager - Infra  157000
240      Roper, Katie          Data Architect  150290
244     Ruiz, Ricardo         IT Manager - DB  148999
243        Roup,Simon         IT Manager - DB  140920
76      Dougall, Eric    IT Manager - Support  138888

**Decision: retain, don't cap or remove.** All 29 flagged salaries belong to senior/leadership positions (CEO, CIO, IT Director, Director of Sales, etc.) — the high salaries are a genuine feature of seniority, not a data entry error. Capping them would understate real pay for leadership roles and distort any compensation analysis. They're flagged here for visibility but left untouched in the cleaned dataset.

## 7. Data Type Correction

In [11]:
df['EmpID'] = df['EmpID'].astype(str)
df['Zip'] = df['Zip'].astype(str).str.zfill(5)
df['Salary'] = df['Salary'].astype(float)

print(df[['EmpID', 'Zip', 'Salary']].dtypes)
print()
print("Zip sample after zero-padding fix:", df['Zip'].head(5).tolist())

EmpID         str
Zip           str
Salary    float64
dtype: object

Zip sample after zero-padding fix: ['01960', '02148', '01810', '01886', '02169']


**Observation:** `EmpID` is now a string, since it's an identifier and should never be summed or averaged. `Zip` is zero-padded back to 5 digits, restoring the leading zeros that `read_csv` had silently dropped for Massachusetts addresses. `Salary` is explicit `float64` for consistent downstream arithmetic.

## 8. Before vs. After Summary

In [12]:
raw_df = pd.read_csv('HRDataset_v14.csv')

summary = pd.DataFrame({
    'Metric': ['Row count', 'Duplicate rows', 'Null cells (total)', 'Date columns as datetime', 'Zip as 5-char string'],
    'Before': [
        len(raw_df),
        raw_df.duplicated().sum(),
        raw_df.isnull().sum().sum(),
        0,
        (raw_df['Zip'].astype(str).str.len() == 5).sum()
    ],
    'After': [
        len(df),
        df.duplicated().sum(),
        df.isnull().sum().sum(),
        4,
        (df['Zip'].str.len() == 5).sum()
    ]
})
summary

                     Metric  Before  After
0                 Row count     311    311
1            Duplicate rows       0      0
2        Null cells (total)     215    215
3  Date columns as datetime       0      4
4      Zip as 5-char string      24    311

**Observation:** Row count is unchanged (311 — no rows were dropped, since nulls in `DateofTermination` and `ManagerID` are legitimate, not errors). Total null cells remain the same for the same reason. The real improvement is structural: all 4 date columns are now proper datetimes instead of strings, and every Zip code is a correctly zero-padded 5-character string instead of a truncated integer.

## 9. Save Cleaned Dataset

In [13]:
df.to_csv('HRDataset_v14_cleaned.csv', index=False)
print("Saved HRDataset_v14_cleaned.csv with shape", df.shape)

Saved HRDataset_v14_cleaned.csv with shape (311, 37)


## 10. Conclusion

This dataset was cleaner than a typical "deliberately messy" practice set, but still required real decisions rather than blanket fixes:

1. **Nulls were mostly meaningful, not missing.** `DateofTermination` and `ManagerID` nulls both represent real-world facts (still employed / no manager) rather than data-entry gaps — imputing them would have introduced false information.
2. **The 2-digit year DOB format was a silent correctness bug**, not just a formatting inconsistency — left unfixed, it would have placed dozens of employees' birth years a century in the future.
3. **Zip code truncation is an easy trap** with `pd.read_csv` — any leading-zero ZIP silently loses its zero unless the column is explicitly cast to string with zero-padding.
4. **Outliers aren't always errors** — the 29 high salaries reflect genuine seniority and were kept unmodified rather than capped, since capping would have quietly falsified executive compensation data.